<a href="https://colab.research.google.com/github/vinkoff/Learner/blob/master/TacticForge_Backend_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TacticForge — OPCG Tactics Engine
## Backend Data Pipeline (Vinoth Selvam)

Run each cell in order. All data is saved to your Google Drive under `OPCG_AI/`.

**Data Sources:**
- Card data → Official Bandai site (`en.onepiece-cardgame.com`)
- Ban list → Official Bandai rules page
- Tournaments & Decklists → Limitless One Piece (`onepiece.limitlesstcg.com`)

---
## Cell 1: Mount Google Drive & Create Project Folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/OPCG_AI'
DB_PATH     = f'{PROJECT_DIR}/data/opcg_engine.db'
EXPORT_DIR  = f'{PROJECT_DIR}/exports'

os.makedirs(f'{PROJECT_DIR}/data',    exist_ok=True)
os.makedirs(EXPORT_DIR,               exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/logs',    exist_ok=True)

print(f"Project folder ready : {PROJECT_DIR}")
print(f"Database path        : {DB_PATH}")
print(f"Exports folder       : {EXPORT_DIR}")

Mounted at /content/drive
Project folder ready : /content/drive/MyDrive/OPCG_AI
Database path        : /content/drive/MyDrive/OPCG_AI/data/opcg_engine.db
Exports folder       : /content/drive/MyDrive/OPCG_AI/exports


---
## Cell 2: Install Dependencies

In [ ]:
!pip install requests beautifulsoup4 pandas numpy lxml --quiet
print("All dependencies installed.")

All dependencies installed.


---
## Cell 3: Create SQLite Database Schema

Creates 7 tables: `cards`, `leaders`, `tournaments`, `decklists`, `decklist_cards`, `ban_list`, `meta_snapshots`

In [ ]:
import sqlite3

def create_schema(db_path):
    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.execute('''CREATE TABLE IF NOT EXISTS cards (
        card_id       TEXT PRIMARY KEY,
        name          TEXT NOT NULL,
        set_code      TEXT,
        card_number   TEXT,
        colors        TEXT,
        card_type     TEXT,
        cost          INTEGER,
        power         INTEGER,
        counter       INTEGER,
        attribute     TEXT,
        effect_text   TEXT,
        block_number  INTEGER,
        is_legal      INTEGER DEFAULT 1,
        rarity        TEXT,
        created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS leaders (
        leader_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        card_id    TEXT UNIQUE,
        life       INTEGER,
        FOREIGN KEY (card_id) REFERENCES cards(card_id)
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS tournaments (
        tournament_id  TEXT PRIMARY KEY,
        name           TEXT,
        date           TEXT,
        location       TEXT,
        format         TEXT,
        tier           TEXT,
        player_count   INTEGER,
        source_url     TEXT
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS decklists (
        decklist_id     INTEGER PRIMARY KEY AUTOINCREMENT,
        tournament_id   TEXT,
        player_name     TEXT,
        placement       INTEGER,
        leader_card_id  TEXT,
        FOREIGN KEY (tournament_id)  REFERENCES tournaments(tournament_id),
        FOREIGN KEY (leader_card_id) REFERENCES cards(card_id)
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS decklist_cards (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        decklist_id  INTEGER,
        card_id      TEXT,
        quantity     INTEGER,
        FOREIGN KEY (decklist_id) REFERENCES decklists(decklist_id),
        FOREIGN KEY (card_id)     REFERENCES cards(card_id)
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS ban_list (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        card_id          TEXT,
        restriction_type TEXT,
        pair_card_id     TEXT,
        effective_date   TEXT,
        source_url       TEXT
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS meta_snapshots (
        snapshot_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        snapshot_date    TEXT,
        leader_card_id   TEXT,
        meta_share       REAL,
        win_rate         REAL,
        top8_appearances INTEGER
    )''')

    conn.commit()
    conn.close()
    print("Database schema created successfully.")

create_schema(DB_PATH)

Database schema created successfully.


---
## Cell 4: Card Scraper (Official Bandai Site)

Scrapes all cards from `en.onepiece-cardgame.com/cardlist/` and saves them to the `cards` table.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}

BANDAI_CARD_LIST = "https://en.onepiece-cardgame.com/cardlist/"

def scrape_bandai_cards():
    cards = []
    try:
        resp = requests.get(BANDAI_CARD_LIST, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        card_items = soup.find_all('dl', class_='modalCol')

        for item in card_items:
            card = {}

            header = item.find('div', class_='cardName')
            if header:
                parts = header.get_text(separator='|', strip=True).split('|')
                card['card_id']   = parts[0].strip() if len(parts) > 0 else ''
                card['card_type'] = parts[1].strip() if len(parts) > 1 else ''
                card['name']      = parts[2].strip() if len(parts) > 2 else ''

            for label, key in [('Cost','cost'), ('Power','power'),
                                ('Counter','counter'), ('Color','colors'),
                                ('Block','block_number'), ('Rarity','rarity')]:
                tag = item.find(string=lambda t: t and label in t)
                if tag:
                    parent  = tag.find_parent()
                    sibling = parent.find_next_sibling() if parent else None
                    if sibling:
                        card[key] = sibling.get_text(strip=True)

            effect = item.find('div', class_='text')
            card['effect_text'] = effect.get_text(strip=True) if effect else ''

            if card.get('card_id'):
                cards.append(card)

        print(f"Scraped {len(cards)} cards from Bandai.")
    except Exception as e:
        print(f"Card scrape error: {e}")
    return cards


def save_cards_to_db(cards, db_path):
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    inserted = 0

    for card in cards:
        try:
            c.execute('''INSERT OR IGNORE INTO cards
                (card_id, name, card_type, cost, power, counter, colors,
                 effect_text, block_number, rarity)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                (
                    card.get('card_id', ''),
                    card.get('name', ''),
                    card.get('card_type', ''),
                    int(card['cost'])         if str(card.get('cost','')).isdigit()         else None,
                    int(card['power'])        if str(card.get('power','')).isdigit()        else None,
                    int(card['counter'])      if str(card.get('counter','')).isdigit()      else None,
                    card.get('colors', ''),
                    card.get('effect_text', ''),
                    int(card['block_number']) if str(card.get('block_number','')).isdigit() else None,
                    card.get('rarity', '')
                ))
            inserted += 1
        except Exception as e:
            print(f"Row insert error ({card.get('card_id')}): {e}")

        if card.get('card_type', '').upper() == 'LEADER':
            c.execute('INSERT OR IGNORE INTO leaders (card_id) VALUES (?)',
                      (card.get('card_id'),))

    conn.commit()
    conn.close()
    print(f"Saved {inserted} cards to database.")


cards = scrape_bandai_cards()
save_cards_to_db(cards, DB_PATH)

Scraped 196 cards from Bandai.
Saved 196 cards to database.


---
## Cell 5: Ban List Scraper (Official Bandai Rules Page)

Scrapes the current ban/restriction list from `en.onepiece-cardgame.com/rules/restriction`.
Also applies the Standard Regulation rule: Block 1 cards marked not legal.

In [ ]:
BANDAI_BANLIST = "https://en.onepiece-cardgame.com/rules/restriction"

def scrape_banlist():
    entries = []
    try:
        resp = requests.get(BANDAI_BANLIST, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        sections = soup.find_all('section')
        for section in sections:
            heading      = section.find(['h2', 'h3'])
            section_label = heading.get_text(strip=True).lower() if heading else ''

            for li in section.find_all('li'):
                card_number = li.find('span', class_='cardNumber')
                if not card_number:
                    card_number = li.find('p')
                card_id = card_number.get_text(strip=True) if card_number else ''
                if not card_id:
                    continue

                if 'pair' in section_label:
                    r_type = 'banned_pair'
                elif 'banned' in section_label:
                    r_type = 'banned'
                else:
                    r_type = 'restricted'

                entries.append({
                    'card_id':          card_id,
                    'restriction_type': r_type,
                    'source_url':       BANDAI_BANLIST
                })

        print(f"Found {len(entries)} ban list entries.")
    except Exception as e:
        print(f"Banlist scrape error: {e}")
    return entries


def save_banlist_to_db(entries, db_path):
    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.execute("DELETE FROM ban_list")

    for entry in entries:
        c.execute('''INSERT INTO ban_list (card_id, restriction_type, source_url)
                     VALUES (?, ?, ?)''',
                  (entry['card_id'], entry['restriction_type'], entry['source_url']))

        if entry['restriction_type'] == 'banned':
            c.execute("UPDATE cards SET is_legal = 0 WHERE card_id = ?", (entry['card_id'],))

    # Standard Regulation: Block 2+ only (as of April 1, 2026)
    c.execute("UPDATE cards SET is_legal = 0 WHERE block_number < 2 OR block_number IS NULL")

    conn.commit()
    conn.close()

    conn2 = sqlite3.connect(db_path)
    legal_count = conn2.execute("SELECT COUNT(*) FROM cards WHERE is_legal = 1").fetchone()[0]
    conn2.close()
    print(f"Ban list saved. Legal cards remaining: {legal_count}")


banlist = scrape_banlist()
save_banlist_to_db(banlist, DB_PATH)

Found 0 ban list entries.
Ban list saved. Legal cards remaining: 0


---
## Cell 6: Tournament & Decklist Scraper (Limitless One Piece)

Collects tournament results and full decklists from `onepiece.limitlesstcg.com`.
Adjust `pages` to control how many tournament pages to scan.

In [ ]:
LIMITLESS_BASE = "https://onepiece.limitlesstcg.com"

def scrape_tournament_list(pages=5):
    tournaments = []
    for page in range(1, pages + 1):
        try:
            url  = f"{LIMITLESS_BASE}/tournaments?page={page}"
            resp = requests.get(url, headers=HEADERS, timeout=20)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'lxml')

            rows = soup.find_all('a', href=lambda h: h and '/tournaments/' in h)
            for row in rows:
                href = row.get('href', '')
                t_id = href.strip('/').split('/')[-1]
                if not t_id.isdigit():
                    continue

                name_el    = row.find(class_='name') or row.find('h3') or row.find('h2')
                date_el    = row.find(class_='date')
                players_el = row.find(class_='players')
                loc_el     = row.find(class_='location')

                player_text  = players_el.get_text(strip=True) if players_el else '0'
                player_count = int(''.join(filter(str.isdigit, player_text)) or 0)

                tournaments.append({
                    'tournament_id': t_id,
                    'name':          name_el.get_text(strip=True) if name_el else f'Tournament {t_id}',
                    'date':          date_el.get_text(strip=True) if date_el else '',
                    'location':      loc_el.get_text(strip=True) if loc_el else '',
                    'player_count':  player_count,
                    'source_url':    f"{LIMITLESS_BASE}{href}"
                })

            print(f"  Page {page}: found {len(rows)} entries")
            time.sleep(1)
        except Exception as e:
            print(f"Tournament list page {page} error: {e}")

    print(f"Total tournaments found: {len(tournaments)}")
    return tournaments


def scrape_decklists(tournament_id):
    url      = f"{LIMITLESS_BASE}/tournaments/{tournament_id}/decklists"
    decklists = []
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        if resp.status_code == 404:
            return []
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        for row in soup.find_all(class_='decklist-row'):
            dl = {'tournament_id': tournament_id, 'cards': []}

            placement_el   = row.find(class_='placement')
            player_el      = row.find(class_='player')
            leader_el      = row.find(class_='leader')

            raw_placement  = placement_el.get_text(strip=True) if placement_el else '0'
            dl['placement']   = int(''.join(filter(str.isdigit, raw_placement)) or 0)
            dl['player_name'] = player_el.get_text(strip=True) if player_el else 'Unknown'

            if leader_el:
                link = leader_el.find('a')
                dl['leader_card_id'] = link['href'].split('/')[-1] if link else ''

            for item in row.find_all(class_='card-item'):
                qty_el    = item.find(class_='quantity')
                card_link = item.find('a')
                qty       = int(qty_el.get_text(strip=True) or 1) if qty_el else 1
                card_id   = card_link['href'].split('/')[-1] if card_link else ''
                if card_id:
                    dl['cards'].append({'card_id': card_id, 'quantity': qty})

            decklists.append(dl)
    except Exception as e:
        print(f"Decklist error for tournament {tournament_id}: {e}")
    return decklists


def save_tournament_and_decklists(tournament, decklists, db_path):
    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.execute('''INSERT OR IGNORE INTO tournaments
        (tournament_id, name, date, location, format, tier, player_count, source_url)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
        (tournament['tournament_id'], tournament.get('name',''),
         tournament.get('date',''), tournament.get('location',''),
         'Standard', 'Regional',
         tournament.get('player_count', 0), tournament.get('source_url','')))

    for dl in decklists:
        c.execute('''INSERT INTO decklists
            (tournament_id, player_name, placement, leader_card_id)
            VALUES (?, ?, ?, ?)''',
            (dl['tournament_id'], dl['player_name'],
             dl['placement'], dl.get('leader_card_id', '')))
        decklist_id = c.lastrowid

        for card in dl['cards']:
            c.execute('''INSERT INTO decklist_cards (decklist_id, card_id, quantity)
                         VALUES (?, ?, ?)''',
                      (decklist_id, card['card_id'], card['quantity']))

    conn.commit()
    conn.close()


# Run — change pages=3 to pages=10 to collect more history
print("Collecting tournament list...")
tournaments = scrape_tournament_list(pages=3)

print("\nCollecting decklists...")
for i, t in enumerate(tournaments):
    decklists = scrape_decklists(t['tournament_id'])
    save_tournament_and_decklists(t, decklists, DB_PATH)
    if (i + 1) % 5 == 0:
        print(f"  Processed {i+1}/{len(tournaments)} tournaments...")
    time.sleep(1.5)

print("Tournament and decklist collection complete.")

  Page 1: found 26 entries
  Page 2: found 26 entries
  Page 3: found 26 entries
Total tournaments found: 75

  Processed 5/75 tournaments...
  Processed 10/75 tournaments...
  Processed 15/75 tournaments...
  Processed 20/75 tournaments...
  Processed 25/75 tournaments...
  Processed 30/75 tournaments...
  Processed 35/75 tournaments...
  Processed 40/75 tournaments...
  Processed 45/75 tournaments...
  Processed 50/75 tournaments...
  Processed 55/75 tournaments...
  Processed 60/75 tournaments...
  Processed 65/75 tournaments...
  Processed 70/75 tournaments...
  Processed 75/75 tournaments...
Tournament and decklist collection complete.


---
## Cell 7: Statistical Analysis Functions

These functions are shared with Elijah's AI layer and Merrill's visualization layer.
They return clean Pandas DataFrames ready for analysis or charting.

In [ ]:
import pandas as pd
import numpy as np

def get_card_frequency(db_path, top_placement=8):
    """Cards ranked by how often they appear in top N finishes."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query('''
        SELECT c.card_id, c.name, c.card_type, c.colors,
               c.cost, c.power, c.effect_text,
               COUNT(dc.decklist_id) AS appearances,
               AVG(d.placement)      AS avg_placement
        FROM cards c
        JOIN decklist_cards dc ON c.card_id = dc.card_id
        JOIN decklists d       ON dc.decklist_id = d.decklist_id
        WHERE d.placement <= ?
        GROUP BY c.card_id
        ORDER BY appearances DESC
    ''', conn, params=(top_placement,))
    conn.close()
    return df


def get_leader_stats(db_path):
    """Win rate, meta share, and top 8 rate per leader."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query('''
        SELECT c.card_id, c.name, c.colors,
               COUNT(d.decklist_id)                              AS total,
               SUM(CASE WHEN d.placement = 1  THEN 1 ELSE 0 END) AS wins,
               SUM(CASE WHEN d.placement <= 8 THEN 1 ELSE 0 END) AS top8
        FROM decklists d
        JOIN cards c ON d.leader_card_id = c.card_id
        GROUP BY c.card_id
        ORDER BY top8 DESC
    ''', conn)
    conn.close()
    total = df['total'].sum()
    df['meta_share'] = (df['total'] / total * 100).round(2) if total > 0 else 0.0
    df['win_rate']   = (df['wins']  / df['total'] * 100).round(2)
    df['top8_rate']  = (df['top8']  / df['total'] * 100).round(2)
    return df


def get_cooccurrence_matrix(db_path, top_n=50):
    """Cards that appear together in top-8 decklists. Input for clustering."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query('''
        SELECT dc.decklist_id, dc.card_id
        FROM decklist_cards dc
        JOIN decklists d ON dc.decklist_id = d.decklist_id
        WHERE d.placement <= 8
    ''', conn)
    conn.close()
    top_cards = df['card_id'].value_counts().head(top_n).index
    df = df[df['card_id'].isin(top_cards)]
    pivot = df.groupby(['decklist_id','card_id']).size().unstack(fill_value=0)
    cooccurrence = pivot.T.dot(pivot)
    return cooccurrence


def get_meta_trends(db_path):
    """Leader appearance counts per tournament date for trend charts."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query('''
        SELECT t.date, c.name AS leader, c.colors,
               COUNT(d.decklist_id) AS appearances
        FROM decklists d
        JOIN tournaments t ON d.tournament_id  = t.tournament_id
        JOIN cards c       ON d.leader_card_id = c.card_id
        WHERE d.placement <= 8
        GROUP BY t.date, c.card_id
        ORDER BY t.date
    ''', conn)
    conn.close()
    return df


def get_db_summary(db_path):
    """Quick row counts for all tables."""
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    for table in ['cards','leaders','tournaments','decklists','decklist_cards','ban_list']:
        count = c.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"  {table:<20} {count:>6} rows")
    legal = c.execute("SELECT COUNT(*) FROM cards WHERE is_legal=1").fetchone()[0]
    print(f"  {'legal cards':<20} {legal:>6} rows")
    conn.close()

print("Analysis functions loaded.")
get_db_summary(DB_PATH)

Analysis functions loaded.
  cards                   131 rows
  leaders                   0 rows
  tournaments              75 rows
  decklists                 0 rows
  decklist_cards            0 rows
  ban_list                  0 rows
  legal cards               0 rows


---
## Cell 8: Export for AI Layer (Elijah) and Visualization Layer (Merrill)

Exports 4 files to `OPCG_AI/exports/`:
- `legal_cards.json` — full card data for the LLM
- `card_frequency_top8.csv` — ranked card appearances
- `leader_stats.csv` — leader win/meta stats
- `cooccurrence_matrix.csv` — card co-occurrence for clustering

In [ ]:
import json

def export_for_ai_and_viz(db_path, export_dir):
    conn = sqlite3.connect(db_path)
    legal_cards = pd.read_sql_query('''
        SELECT card_id, name, card_type, colors, cost, power,
               counter, attribute, effect_text, block_number
        FROM cards WHERE is_legal = 1
    ''', conn)
    conn.close()

    card_freq    = get_card_frequency(db_path, top_placement=8)
    leader_stats = get_leader_stats(db_path)
    cooccurrence = get_cooccurrence_matrix(db_path)

    legal_cards.to_json(f'{export_dir}/legal_cards.json',    orient='records', indent=2)
    card_freq.to_csv(   f'{export_dir}/card_frequency_top8.csv', index=False)
    leader_stats.to_csv(f'{export_dir}/leader_stats.csv',        index=False)
    cooccurrence.to_csv(f'{export_dir}/cooccurrence_matrix.csv')

    print(f"Exported {len(legal_cards)} legal cards  →  legal_cards.json")
    print(f"Exported card frequency                  →  card_frequency_top8.csv")
    print(f"Exported leader stats                    →  leader_stats.csv")
    print(f"Exported co-occurrence matrix            →  cooccurrence_matrix.csv")
    print(f"All files saved to: {export_dir}")


export_for_ai_and_viz(DB_PATH, EXPORT_DIR)

ValueError: no types given

---
## Cell 9: Full Pipeline Runner

Run this single cell to refresh all data in one go.
Use this when a new set releases or after a major tournament weekend.

In [ ]:
def run_full_pipeline(db_path=DB_PATH, export_dir=EXPORT_DIR, tournament_pages=3):
    print("=" * 50)
    print("TACTICFORGE — Full Pipeline Run")
    print("=" * 50)

    print("\n[1/5] Setting up database schema...")
    create_schema(db_path)

    print("\n[2/5] Collecting card data from Bandai...")
    save_cards_to_db(scrape_bandai_cards(), db_path)

    print("\n[3/5] Collecting ban list from Bandai...")
    save_banlist_to_db(scrape_banlist(), db_path)

    print(f"\n[4/5] Collecting tournaments (pages={tournament_pages})...")
    for t in scrape_tournament_list(pages=tournament_pages):
        dls = scrape_decklists(t['tournament_id'])
        save_tournament_and_decklists(t, dls, db_path)
        time.sleep(1.5)

    print("\n[5/5] Exporting data for AI and visualization layers...")
    export_for_ai_and_viz(db_path, export_dir)

    print("\n" + "=" * 50)
    print("Pipeline complete. Database summary:")
    get_db_summary(db_path)
    print("=" * 50)


# Uncomment to run a full refresh:
# run_full_pipeline(tournament_pages=5)
print("Pipeline runner ready. Call run_full_pipeline() to execute.")